In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import stats
from src import plt_func

### data - enrichment 

In [ ]:
# report enrichment summary from list of files
def get_enrichment_data_from_files(nseq_list, *enrichment_files):
    df = pd.DataFrame([])
    for f in enrichment_files:
        df_i = pd.read_csv(f, sep="\t", index_col=0)
        df = pd.concat([df, df_i])
    
    df.columns = df.columns.astype(int)
    df = df.reindex(sorted(df.columns), axis=1)
    
    return([np.array(nseq_list), df.mean().values, df.sem().values])

In [ ]:
# report enrichment of a sample, spp vs crowd, spp ctrl vs crowd
def get_enrichment_data_sample(data_type, nseq_list):
    full = pd.DataFrame([])
    
    data = get_enrichment_data_from_files(nseq_list, "/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/RankCrowdness_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1], data[2]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "C-score"
    full = pd.concat([full, full_df])

    data = get_enrichment_data_from_files(nseq_list, "/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/RankSPP_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1], data[2]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "SPP"
    full = pd.concat([full, full_df])

    data = get_enrichment_data_from_files(nseq_list, "/home/jg2447/slayman/motif_inference/result/EnrichmentGenomeShuffle/%s/enrichment/RankCrowdness_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1], data[2]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "C-score Shuffled"
    full = pd.concat([full, full_df])
    
    data = get_enrichment_data_from_files(nseq_list, "/home/jg2447/slayman/motif_inference/result/EnrichmentGenomeShuffle/%s/enrichment/RankSPP_100.txt" % data_type)
    full_df = pd.DataFrame([data[0], data[1], data[2]]).transpose()
    full_df["sample"] = data_type.replace("human", "")
    full_df["rank"] = "SPP Shuffled"
    full = pd.concat([full, full_df])
    
    return(full)

In [ ]:
nseq_list_sub = [50]+list(range(250, 3250, 250))
nseq_list = list(range(50, 3010, 10))

full1 = get_enrichment_data_sample("fly", nseq_list)
full2 = get_enrichment_data_sample("worm", nseq_list)
full3 = get_enrichment_data_sample("humanGM12878", nseq_list)
full4 = get_enrichment_data_sample("humanK562", nseq_list)

enrichment_data = pd.concat([full1, full2, full3, full4]).reset_index(drop=True)
enrichment_data.columns = ["top", "enrichment", "err", "sample", "rank"]

enrichment_data.to_csv("fig2_data_enrichment.txt", sep="\t", index=False)

# subset
enrichment_data_subset = enrichment_data[enrichment_data['top'].isin(nseq_list_sub)].reset_index(drop=True)
# jit for plot
enrichment_data_subset.loc[enrichment_data_subset["rank"] == "C-score", "top"] += 10
enrichment_data_subset.loc[enrichment_data_subset["rank"] == "C-score Shuffled", "top"] += 10

enrichment_data_subset.to_csv("fig2_data_enrichment_subset.txt", sep="\t", index=False)

### data - enrichment change rank

In [ ]:
def get_enrichment_data_sample_rank(data_type, rank_list):
    data_val = []
    data_mean = []
    
    for rank in rank_list:
        df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/%s_100.txt" % (data_type, rank), sep="\t", index_col=0)
        data_mean.append(df["250"].mean())
        data_val.append(df["250"].values)
        if rank == "RankSPP":
            spp_val = df["250"].values
            spp_mean = df["250"].mean()
        elif rank == "RankCrowdness":
            crowd_val = df["250"].values
            crowd_mean = df["250"].mean()

    enrichment_improve_data = (np.array(data_mean) - spp_mean)/spp_mean

    spp_p_list = []
    crowd_p_list = []
    for i in range(len(data_val)):
        spp_p_list.append(stats.sig_test_wilcoxon(data_val[i], spp_val))
        crowd_p_list.append(stats.sig_test_wilcoxon(data_val[i], crowd_val))

    enrichment_improve_p = []
    for i in range(len(spp_p_list)):
        if spp_p_list[i] < 0.05 and crowd_p_list[i] < 0.05:
            enrichment_improve_p.append("*+")
        elif spp_p_list[i] < 0.05 and crowd_p_list[i] >= 0.05:
            enrichment_improve_p.append("*")
        elif spp_p_list[i] >= 0.05 and crowd_p_list[i] < 0.05:
            enrichment_improve_p.append("+")
        elif spp_p_list[i] >= 0.05 and crowd_p_list[i] >= 0.05:
            enrichment_improve_p.append("")

    return((enrichment_improve_data, np.array(enrichment_improve_p)))

In [ ]:
alpha_list_para = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list_para = ["RankLinear_%.1f" % i for i in alpha_list_para]
rank_list = ["RankSPP"]
rank_list.extend(rank_list_para)
rank_list.extend(["RankCrowdness"])

data1, p1 = get_enrichment_data_sample_rank("fly", rank_list)
df1 = pd.DataFrame([rank_list, data1, p1, ["fly"]*len(rank_list)]).T
data2, p2 = get_enrichment_data_sample_rank("worm", rank_list)
df2 = pd.DataFrame([rank_list, data2, p2, ["worm"]*len(rank_list)]).T
data3, p3 = get_enrichment_data_sample_rank("humanGM12878", rank_list)
df3 = pd.DataFrame([rank_list, data3, p3, ["humanGM12878"]*len(rank_list)]).T
data4, p4 = get_enrichment_data_sample_rank("humanK562", rank_list)
df4 = pd.DataFrame([rank_list, data4, p4, ["humanK562"]*len(rank_list)]).T

enrichment_data_rank = pd.concat([df1, df2, df3, df4]).reset_index(drop=True)
enrichment_data_rank.columns = ["rank", "enrichment", "p", "sample"]

enrichment_data_rank.to_csv("./fig2_data_enrichment_rank.txt", sep="\t", index=False)

### data - enrichment change length

In [ ]:
def get_data_sample_length(data_type, nbp_list):
    enrichment_data_length = []
    
    for nn in nbp_list:
        df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/%s/enrichment/RankSPP_%d.txt" % (data_type, nn), sep="\t", index_col=0)
        enrichment_data_length.append(df["250"].mean())
        if nn == 100:
            enrichment_reference = df["250"].mean()
    enrichment_improve_data = (np.array(enrichment_data_length) - enrichment_reference)/enrichment_reference
    return(enrichment_improve_data)

In [ ]:
nbp_list = [10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

datalen1 = get_data_sample_length("fly", nbp_list)
df1 = pd.DataFrame([nbp_list, datalen1, ["fly"]*len(nbp_list)]).T
datalen2 = get_data_sample_length("worm", nbp_list)
df2 = pd.DataFrame([nbp_list, datalen2, ["worm"]*len(nbp_list)]).T
datalen3 = get_data_sample_length("humanGM12878", nbp_list)
df3 = pd.DataFrame([nbp_list, datalen3, ["humanGM12878"]*len(nbp_list)]).T
datalen4 = get_data_sample_length("humanK562", nbp_list)
df4 = pd.DataFrame([nbp_list, datalen4, ["humanK562"]*len(nbp_list)]).T

enrichment_data_length = pd.concat([df1, df2, df3, df4]).reset_index(drop=True)
enrichment_data_length.columns = ["nbp", "enrichment", "sample"]
enrichment_data_length.to_csv("./fig2_data_enrichment_length.txt", sep="\t", index=False)

### plot

In [ ]:
# plot a single line with error bar
def plt_enrichment_line(ax, enrichment_data, enrichment_data_subset, color, sample):

    # plot lines
    ax.plot("top", "enrichment", "-", data=enrichment_data.query("sample == '%s' and rank == 'C-score'" % sample), color=color[0], lw=0.75)
    ax.plot("top", "enrichment", "-", data=enrichment_data.query("sample == '%s' and rank == 'SPP'" % sample), color=color[1], lw=0.75)
    ax.plot("top", "enrichment", "--", data=enrichment_data.query("sample == '%s' and rank == 'C-score Shuffled'" % sample), color=color[2], lw=0.75)
    ax.plot("top", "enrichment", "--", data=enrichment_data.query("sample == '%s' and rank == 'SPP Shuffled'" % sample), color=color[3], lw=0.75)
    
    # xaxis
    ax.xaxis.set_ticks([0, 1000, 2000, 3000])
    ax.tick_params(labelsize=x_tick_label_fs, pad=1, length=2.5, width=0.5)
    
    # box
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['left'].set_linewidth(0.5)
        
    for i,r in enumerate(["C-score", "SPP", "C-score Shuffled", "SPP Shuffled"]):
        # plot error bar
        (_, caps, _) = ax.errorbar(
            x = enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["top"], 
            y = enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["enrichment"], 
            yerr = enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["err"], 
            fmt="o", ms=0.75, elinewidth=0.75, capsize=1, color=color[i])
        for cap in caps:
            cap.set_markeredgewidth(0)

#         # plot error bar ribbon
#         y1 = enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["enrichment"].values + enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["err"].values
#         y2 = enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["enrichment"].values - enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["err"].values
#         ax.fill_between(
#             x = enrichment_data_subset.query("sample == '%s' and rank == '%s'" % (sample,r))["top"],
#             y1 = y1,
#             y2 = y2,
#             color=color[i])
    return

In [ ]:
def plt_heatmap_rank(data, data_p, axs):
    # axs: list of ax: 0 - spp, 1 - BC, 2 - C, 3 - cbar
    
    # SPP part
    ax1 = sns.heatmap(
        pd.DataFrame([data]).iloc[:, 0:1],
        ax=axs[0],
        annot=pd.DataFrame([data_p]).iloc[:, 0:1],
        cbar=False,
        vmax=abs(data).max(), vmin=-abs(data).max(), center=0, cmap=cmap, annot_kws={"fontsize":6}, fmt="s", linewidths=.05)
    # BC-score part
    ax2 = sns.heatmap(
        pd.DataFrame([data]).iloc[:, 1:-1],
        ax=axs[1],
        annot=pd.DataFrame([data_p]).iloc[:, 1:-1],
        cbar_ax=axs[3], cbar_kws={"orientation": "horizontal"}, # add color bar
        vmax=abs(data).max(), vmin=-abs(data).max(), center=0, cmap=cmap, annot_kws={"fontsize":6}, fmt="s", linewidths=.05)
    # C-score part
    ax3 = sns.heatmap(
        pd.DataFrame([data]).iloc[:, -1:],
        ax=axs[2],
        annot=pd.DataFrame([data_p]).iloc[:, -1:],
        cbar=False,
        vmax=abs(data).max(), vmin=-abs(data).max(), center=0, cmap=cmap, annot_kws={"fontsize":6}, fmt="s", linewidths=.05)
       
    # no tick/label
    ax1 = plt_func.rm_axis_label(ax1, "y")    
    ax2 = plt_func.rm_axis_label(ax2, "y")    
    ax3 = plt_func.rm_axis_label(ax3, "y")
    ax1 = plt_func.rm_axis_label(ax1, "x")
    ax2 = plt_func.rm_axis_label(ax2, "x")
    ax3 = plt_func.rm_axis_label(ax3, "x")
    ax1.yaxis.set_ticks([0.5])

    # no color bar tick/label
    ax_cbar = plt_func.rm_axis_label(axs[3], "x")
    
    # color bar label
    ax_cbar.annotate("%.2f" % (-abs(data).max()), xy=(-15,0.2), fontsize=legend_fs, xycoords="axes pixels")
    ax_cbar.annotate("%.2f" % (abs(data).max()), xy=(59,0.2), fontsize=legend_fs, xycoords="axes pixels")
    
    return((ax1, ax2, ax3))

In [ ]:
def plt_heatmap_length(ax, data, vmax, ax_cbar):
    ax = sns.heatmap(
        data,
        ax=ax,
        annot=data,
        cbar_ax=ax_cbar,
        cbar_kws={"orientation":"horizontal"},
        vmax=vmax, vmin=-vmax, center=0, cmap=cmap, annot_kws={"fontsize":5}, fmt=".2f")
    
    # no tick/label
    plt_func.rm_axis_label(ax, "x")
    plt_func.rm_axis_label(ax, "y")
    ax.yaxis.set_ticks([0.5])
    
    plt_func.rm_axis_label(ax_cbar, "x")
    return

In [ ]:
#### plot

figsize = (3.2, 6.4)

panel_number_fs = 8
x_tick_label_fs = 6
y_tick_label_fs = 6
x_label_fs = 7
y_label_fs = 7
title_fs = 7
legend_fs = 5

color = ["#c2d2ed", "#ed6a6d", "#c2d2ed", "#ed6a6d"]
cmap = "RdBu"

sns.set_context("paper")

fig, axs = plt.subplots(
    nrows=2, ncols=1, 
    sharex=False, sharey=False,
    gridspec_kw={'hspace': 0.25, "height_ratios":[3.2, 3.2]},
    figsize=figsize, dpi=300)
ax_ad = axs[0]
ax_ef = axs[1]
ax_ad.axis('off') # upper: A-D
ax_ef.axis('off') # lower: E-F

######## A-D

# get data
enrichment_data = pd.read_csv("./fig2_data_enrichment.txt", sep="\t")
enrichment_data_subset = pd.read_csv("./fig2_data_enrichment_subset.txt", sep="\t")

# grid
gs = mpl.gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=ax_ad, hspace=0.4, wspace=0.3)

#### A
ax = fig.add_subplot(gs[0,0])
plt_enrichment_line(ax, enrichment_data, enrichment_data_subset, color, sample="fly")
ax.yaxis.set_ticks([0.07, 0.14, 0.21, 0.28])
ax.set_ylim([0.07,0.28])
ax.set_ylabel("Motif enrichment", fontsize=y_label_fs)
ax.set_title("Fly", fontsize=title_fs)
ax.text(-0.3, 1.07, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### legend for A-D
ax_cbar = ax
ax_cbar.legend(
    handles=[
        (ax.lines[0], ax.lines[4]),
        (ax.lines[1], ax.lines[7]),
        (ax.lines[2], ax.lines[10]),
        (ax.lines[3], ax.lines[13])],
    labels=["C-score","SPP","C-score Ctrl","SPP Ctrl"],
    loc="upper left", fontsize=legend_fs, ncol=4, handlelength=2, bbox_to_anchor=(0,1.42))

#### B
ax = fig.add_subplot(gs[0,1])
plt_enrichment_line(ax, enrichment_data, enrichment_data_subset, color, sample="worm")
ax.yaxis.set_ticks([0,0.08,0.16,0.24])
ax.set_ylim([0,0.24])
ax.set_title("Worm", fontsize=title_fs)
ax.text(-0.2, 1.07, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### C
ax = fig.add_subplot(gs[1,0])
plt_enrichment_line(ax, enrichment_data, enrichment_data_subset, color, sample="GM12878")
ax.yaxis.set_ticks([0.1, 0.25, 0.4, 0.55])
ax.set_ylim([0.075,0.575])
ax.set_ylabel("Motif enrichment", fontsize=y_label_fs)
ax.set_xlabel("Binding sites ranking cutoff", fontsize=x_label_fs)
ax.set_title("GM12878", fontsize=title_fs)
ax.text(-0.3, 1.07, "C", transform=ax.transAxes, size=panel_number_fs, weight='bold')

#### D
ax = fig.add_subplot(gs[1,1])
plt_enrichment_line(ax, enrichment_data, enrichment_data_subset, color, sample="K562")
ax.yaxis.set_ticks([0.1, 0.25, 0.4, 0.55])
ax.set_ylim([0.075,0.575])
ax.set_xlabel("Binding sites ranking cutoff", fontsize=x_label_fs)
ax.set_title("K562", fontsize=title_fs)
ax.text(-0.2, 1.07, "D", transform=ax.transAxes, size=panel_number_fs, weight='bold')

sns.despine(trim=True)

################ E-F
gs = mpl.gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=ax_ef, hspace=0.3, height_ratios=[1.5,1])
ax_e = fig.add_subplot(gs[0,0])
ax_f = fig.add_subplot(gs[1,0])
ax_e.axis('off') # E
ax_f.axis('off') # F

################ E

# data
enrichment_data_rank = pd.read_csv("./fig2_data_enrichment_rank.txt", sep="\t", keep_default_na=False)

# grid
gs = mpl.gridspec.GridSpecFromSubplotSpec(
    9, 5, 
    subplot_spec=ax_e, 
    hspace=0.4,
    wspace=0.05,
    height_ratios=[0.3,1,0.3,1,0.3,1,0.3,1,2],
    width_ratios=[1,6,7,6,1])

## fly
ax1 = fig.add_subplot(gs[1,0]) # spp
ax2 = fig.add_subplot(gs[1,1:4]) # bc-score
ax3 = fig.add_subplot(gs[1,4]) # c-score
ax_cbar = fig.add_subplot(gs[0,2])
data = enrichment_data_rank.query("sample == 'fly'")
ax1, ax2, ax3 = plt_heatmap_rank(data["enrichment"].values, data["p"].values, (ax1, ax2, ax3, ax_cbar))
ax1.yaxis.set_ticklabels(["Fly"], rotation=0, fontsize=y_tick_label_fs)
ax1.text(-1, 1.1, "E", transform=ax1.transAxes, size=panel_number_fs, weight='bold')

## worm
ax1 = fig.add_subplot(gs[3,0])
ax2 = fig.add_subplot(gs[3,1:4])
ax3 = fig.add_subplot(gs[3,4])
ax_cbar = fig.add_subplot(gs[2,2])
data = enrichment_data_rank.query("sample == 'worm'")
ax1, ax2, ax3 = plt_heatmap_rank(data["enrichment"].values, data["p"].values, (ax1, ax2, ax3, ax_cbar))
ax1.yaxis.set_ticklabels(["Worm"], rotation=0, fontsize=y_tick_label_fs)

## GM12878
ax1 = fig.add_subplot(gs[5,0])
ax2 = fig.add_subplot(gs[5,1:4])
ax3 = fig.add_subplot(gs[5,4])
ax_cbar = fig.add_subplot(gs[4,2])
data = enrichment_data_rank.query("sample == 'humanGM12878'")
ax1, ax2, ax3 = plt_heatmap_rank(data["enrichment"].values, data["p"].values, (ax1, ax2, ax3, ax_cbar))
ax1.yaxis.set_ticklabels(["GM12878"], rotation=0, fontsize=y_tick_label_fs)

## K562
ax1 = fig.add_subplot(gs[7,0])
ax2 = fig.add_subplot(gs[7,1:4])
ax3 = fig.add_subplot(gs[7,4])
ax_cbar = fig.add_subplot(gs[6,2])
data = enrichment_data_rank.query("sample == 'humanK562'")
ax1, ax2, ax3 = plt_heatmap_rank(data["enrichment"].values, data["p"].values, (ax1, ax2, ax3, ax_cbar))
ax1.yaxis.set_ticklabels(["K562"], rotation=0, fontsize=y_tick_label_fs)

# X ticks and labels
ax1.xaxis.set_ticks([0.5])
ax1.xaxis.set_ticklabels(["SPP"], fontsize=x_tick_label_fs-1, rotation=90)

ax2.xaxis.set_ticks(np.arange(0.5,19.5,1))
ax2.xaxis.set_ticklabels(
    ["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0", "2.0", "3.0", "4.0", "5.0", "6.0", "7.0", "8.0", "9.0", "10.0"],
    fontsize=x_tick_label_fs-1, rotation=90)

ax3.xaxis.set_ticks([0.5])
ax3.xaxis.set_ticklabels(["C-score"], fontsize=x_tick_label_fs-1, rotation=90)

for axx in [ax1, ax2, ax3]:
    axx.xaxis.set_tick_params(size=2.5, pad=1, color="#757575")

ax2.set_xlabel("Binding site ranking method", fontsize=x_label_fs, labelpad=7)

# BC-score annotation
ax_annot = fig.add_subplot(gs[8,1:4])
ax_annot.plot([1,21], [1,1], ",", color="white")
ax_annot.set_ylim([0,2])
ax_annot.set_xlim([0,21])    
ax_annot.annotate("", (0.5, 0.4), (5.5, 0.4), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", (15.5, 0.4), (20.5, 0.4), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", (0.5, 0), (0.5, 0.8), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", (20.5, 0), (20.5, 0.8), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("BC-score with different weight", (5.5, 0.25), (5.5, 0.25), rotation=0, size=5)
plt_func.rm_axis_label(ax_annot, "x")
ax_annot.axis('off')


################ F

# data
enrichment_data_length = pd.read_csv("./fig2_data_enrichment_length.txt", sep="\t")

# grid
gs = mpl.gridspec.GridSpecFromSubplotSpec(
    nrows=6, ncols=5, 
    subplot_spec=ax_f,
    wspace=0.1,
    hspace=0.1,
    width_ratios=[0.15,1,1,1,0.15],
    height_ratios=[0.3, 0.2, 1, 1, 1, 1])

ax_cbar = fig.add_subplot(gs[0, 2])
vmax = enrichment_data_length['enrichment'].abs().max()

## fly
ax = fig.add_subplot(gs[2, 1:4])
plt_heatmap_length(ax, enrichment_data_length.query("sample == 'fly'")["enrichment"].values.reshape(1,-1), vmax, ax_cbar)
ax.yaxis.set_ticklabels(["Fly"], fontsize=y_tick_label_fs, rotation=0)

ax.text(-0.15, 1.15, "F", transform=ax.transAxes, size=panel_number_fs, weight='bold')

## worm
ax = fig.add_subplot(gs[3, 1:4])
plt_heatmap_length(ax, enrichment_data_length.query("sample == 'worm'")["enrichment"].values.reshape(1,-1), vmax, ax_cbar)
ax.yaxis.set_ticklabels(["Worm"], fontsize=y_tick_label_fs, rotation=0)
    

## GM12878
ax = fig.add_subplot(gs[4, 1:4])
plt_heatmap_length(ax, enrichment_data_length.query("sample == 'humanGM12878'")["enrichment"].values.reshape(1,-1), vmax, ax_cbar)
ax.yaxis.set_ticklabels(["GM12878"], fontsize=y_tick_label_fs, rotation=0)
    

## K562
ax = fig.add_subplot(gs[5, 1:4])
plt_heatmap_length(ax, enrichment_data_length.query("sample == 'humanK562'")["enrichment"].values.reshape(1,-1), vmax, ax_cbar)
ax.yaxis.set_ticklabels(["K562"], fontsize=y_tick_label_fs, rotation=0)

# x ticks and labels
ax.xaxis.set_ticklabels([20,40,80,120,160,200,240,280,320,360,400], rotation=0, fontsize=x_tick_label_fs)
ax.xaxis.set_tick_params(size=2.5, color="#757575", pad=1)
ax.set_xlabel("Binding site length", fontsize=7)

ax_cbar.annotate("%.2f" % (-vmax), xy=(-15,0.4), fontsize=legend_fs, xycoords="axes pixels")
ax_cbar.annotate("%.2f" % (vmax), xy=(52,0.4), fontsize=legend_fs, xycoords="axes pixels")

# plt.show()
plt.savefig("./fig2.pdf", dpi='figure', bbox_inches="tight")